この教材では、コンピュータがすべてを 0 と 1、つまりビットで扱っていること、そしてその「バイナリ」の知識が 情報セキュリティ にどうつながるかを学びます。

- 前半はバイナリの土台: 2進数・16進数・ビット演算・文字コード・バイト列・パケット
- 後半は情報セキュリティ: 通信を脅かす 盗聴・改ざん・なりすまし と、その対策である共通鍵暗号・公開鍵暗号・ハッシュ・デジタル署名

最後の総仕上げでは、これらを組み合わせて 「届いたパケットのマスクを解き、改ざんを検知し、中身を読み取る」 ミニ復号ツールを自分で完成させます。

`____` や `pass` の部分は、自分で書き換える場所です。教材パートの実行例は、まず読んで・動かして・仕組みを確かめてから、練習問題に進みましょう。

# 前半: バイナリの土台

## 2進数と16進数

人は普段 0〜9 の10進数を使いますが、コンピュータは内部を 0 と 1 だけの2進数で表しています。2進数は桁が長く読みにくいので、0〜9 と a〜f を使う16進数もよく使われます。

Python では数値の前に記号を付けて直接書けます。

- `0b` … 2進数。b は binary の頭文字。各桁は 2 の n 乗。`0b1111` = 8+4+2+1 = 15
- `0x` … 16進数。x は hexadecimal から。各桁は 16 の n 乗。1桁がちょうど4ビットに対応し、8ビットの1バイトを2桁で表せて便利

In [ ]:
print(0b100)              # 4
print(0b1111, "=", 8 + 4 + 2 + 1)   # 15 = 15
print(0x41)               # 65
print(0xff)               # 255  ← 1バイトで表せる最大値

# 0b / 0x で書いても中身はただの整数。10進数として表示される
print(0b1010 == 10)       # True
print(0x0a == 10)         # True

## `bin` / `hex` / `int` で基数を変換する

`0b1010` のように 手で書く のではなく、変数の値 を2進や16進の表示に変えたいときは変換関数を使います。

- `bin(x)` … 整数を「2進数の文字列」に。`bin(10)` は `'0b1010'`
- `hex(x)` … 整数を「16進数の文字列」に。`hex(10)` は `'0xa'`
- `int("1010", 2)` / `int("ff", 16)` … 文字列を整数に戻す。第2引数が基数

In [ ]:
print(bin(10))          # 0b1010   ← 文字列
print(hex(255))         # 0xff     ← 文字列
print(int("1010", 2))   # 10
print(int("ff", 16))    # 255

## 練習問題1: 10・50・100 を3つの基数で表す

`for` ループで、各数を 10進・2進・16進 の表示にして出力してください。

ヒント: 変数の値を2進・16進の文字列にするには `bin()` / `hex()` を使います。

期待される出力:
```
10 -> 10進:10  2進:0b1010  16進:0xa
50 -> 10進:50  2進:0b110010  16進:0x32
100 -> 10進:100  2進:0b1100100  16進:0x64
```

In [ ]:
for n in [10, 50, 100]:
    decimal = ____   # n の10進数。そのまま
    binary  = ____   # n を2進数の文字列に
    hexad   = ____   # n を16進数の文字列に
    print("{} -> 10進:{}  2進:{}  16進:{}".format(n, decimal, binary, hexad))

## ビット演算子

2進数で表した数の 各ビット に対して計算します。

| 演算子 | 名前 | はたらき |
|:---:|---|---|
| `&` | AND | 論理積。両方が1のとき1 |
| `\|` | OR | 論理和。どちらかが1なら1 |
| `^` | XOR | 排他的論理和。違えば1、同じなら0 |
| `~` | NOT | 0と1を反転 |
| `<<` | 左シフト | ビットを左へ。1つで×2 |
| `>>` | 右シフト | ビットを右へ。1つで÷2 |

特に XOR `^` は後半の暗号で主役になります。「同じ値で2回XORすると元に戻る」性質を覚えておきましょう。

In [ ]:
print(bin(0b1100 | 0b1010))   # OR  -> 0b1110
print(bin(0b1100 & 0b1010))   # AND -> 0b1000
print(bin(0b1100 ^ 0b1010))   # XOR -> 0b110
print(1 << 4)                 # 左シフト -> 16
print(240 >> 4)               # 右シフト -> 15
print(5 ^ 3 ^ 3)              # 同じ値で2回XORすると元の 5 に戻る

## 練習問題2: ビット演算とマスク

(1) `a = 12`, `b = 25` の OR と AND を求めてください。
(2) `245`、2進数で `0b11110101` から 上位4ビット と 下位4ビット を取り出してください。

ヒント: 取り出したいビットだけを1にした マスク と `&` を取ります。上位4ビットならマスクは `0b11110000` です。

期待される出力:
```
OR: 29
AND: 8
上位: 0b11110000
下位: 0b101
```

In [ ]:
a = 12
b = 25
print("OR:", ____)
print("AND:", ____)

octet4 = 0b11110101
high4 = ____   # 上位4ビット
low4  = ____   # 下位4ビット
print("上位:", bin(high4))
print("下位:", bin(low4))

## 文字コード: 文字 ⇔ 数

コンピュータは数しか扱えないので、文字にも数を割り当てます。英数記号なら ASCII、世界中の文字を含むなら Unicode です。`ord()` で文字→数、`chr()` で数→文字。

In [ ]:
print(ord('A'))                 # 65
print(hex(ord('A')))            # 0x41
print(chr(65))                  # A
print([ord(c) for c in "Hi"])   # [72, 105]

## 練習問題3: 文字コードのリストを文字列に戻す

ユニコードの数のリストを文字列にして返す関数 `convert_to_str` を完成させてください。

ヒント: 各数を `chr()` で文字にしてつなげます。

期待される動作: `convert_to_str([109,105,116,115,117,121,97])` → `'mitsuya'`

In [ ]:
def convert_to_str(charcodes):
    # ここに書いてください
    pass

print(convert_to_str([109, 105, 116, 115, 117, 121, 97]))   # mitsuya

## バイト列 `bytes`

ビットを8個まとめた 0〜255 の値をとる バイト が実データの基本単位です。ファイルもネットワークのパケットも正体は バイト列。

- `bytes([192,168,0,1])` … 数のリストから作る / `.hex()` … 16進文字列へ / `bytes.fromhex(...)` … 逆
- `"文字".encode("utf-8")` / `バイト列.decode("utf-8")` … 文字列 ⇔ バイト列

In [ ]:
data = bytes([192, 168, 0, 1])
print(data[0])                   # 192
print(data.hex())                # c0a80001
print(bytes.fromhex("c0a80001")) # b'\xc0\xa8\x00\x01'
print("あ".encode("utf-8").hex())            # e38182
print(bytes.fromhex("e38182").decode("utf-8"))  # あ

## base64 エンコード

画像や暗号文のような バイナリ を、メールやURLのような「文字しか通らない場所」で運びたいことがあります。バイナリを 文字だけ で表し直すのが base64 です。

※ base64 は 暗号ではありません。鍵なしで誰でも元に戻せます。あくまで「運ぶための変換」です。後半の暗号と混同しないよう注意してください。

In [ ]:
import base64
memo_bytes = "秘密のメモ".encode("utf-8")
enc = base64.b64encode(memo_bytes)
print(enc.decode())                          # base64の文字列
dec = base64.b64decode(enc)
print(dec.decode("utf-8")) # 秘密のメモ

## 練習問題4: 多重base64をデコードする

文字列 `data` は base64 で 3回 エンコードされています。元に戻してください。

ヒント: `base64.b64decode(...)` を3回。`bytes` と `str` の変換に注意し、最後は `.decode()` で文字列に戻します。

期待される出力: `ひみつ`

In [ ]:
import base64
data = "TkRSSGVUUTBSeTgwTkVkcg=="

result = data.encode()
# ここで3回デコードする


print(result.decode())   # ひみつ

## `struct` でパケットを組む・解く

通信では複数の値を 決まった並びのバイト列、つまりパケットにまとめて送ります。`struct` で数値とバイト列を相互変換します。

書式 `">BHhh"` の意味:
- `>` … ビッグエンディアン。上位バイトが先で、ネットワークの標準
- `B` … 1バイト・符号なし / `H` … 2バイト・符号なし / `h` … 2バイト・符号あり。小文字が符号ありです

例として「バージョン・ID・x座標・y座標」を1つのパケットにします。対戦ゲームの通信そのものです。

In [ ]:
import struct
packet = struct.pack(">BHhh", 1, 7, 100, -20)  # version=1, id=7, x=100, y=-20
print(packet.hex())                            # 0100070064ffec の7バイト

version, pid, x, y = struct.unpack(">BHhh", packet)
print(version, pid, x, y)                      # 1 7 100 -20
# y=-20 が ffec になるのは「2の補数」。座標は負もありうるので符号ありの小文字 h を使う

# 後半: 情報セキュリティ

ネットワークを流れるデータ、つまりバイト列は、放っておくと次の3つの脅威にさらされます。

| 脅威 | どういうこと | 主な対策 |
|---|---|---|
| 盗聴 | 通信を第三者に覗かれる | 暗号化。共通鍵／公開鍵 |
| 改ざん | 途中で中身をすり替えられる | ハッシュ／デジタル署名 |
| なりすまし | 別人になりすまして送る | デジタル署名／証明書 |

後半では、この対策技術を一つずつ手を動かして体験します。

## 対策1: 共通鍵暗号で盗聴を防ぐ

共通鍵暗号 は、暗号化と復号に 同じ鍵 を使う方式です。処理が速いのが利点。

前半で学んだ XOR暗号がまさに共通鍵の最小例 です。`a ^ 鍵 ^ 鍵 == a` なので、同じ鍵でXORすれば戻ります。

> この XOR は遊びではありません。次回学ぶ WebSocket の通信では、送信データが XOR で マスク されています。

弱点: 送信者と受信者が 同じ鍵を共有 する必要があり、「その鍵をどうやって安全に相手へ渡すか」という鍵配送問題が課題です。

In [ ]:
def xor_cipher(data, key):
    """共通鍵暗号。暗号化と復号の兼用。bytes の各バイトを key とXORする。"""
    return bytes(b ^ key for b in data)

secret = xor_cipher(b"HELLO", 0x2a)
print(secret.hex())                       # 暗号化された読めないバイト列
plain = xor_cipher(secret, 0x2a)
print(plain.decode())  # HELLO ← 同じ鍵で復号

## 練習問題5: XOR暗号を総当りで破る

暗号文 `secret` は、ある 0〜255 の1バイトの鍵でXORされています。鍵は分かりません。
すべての鍵を試して、復号結果が `FLAG{` で始まる正しい鍵を見つけ、フラグを表示してください。

ヒント: `for key in range(256):` で全部試し、`.startswith(b"FLAG")` で判定します。共通鍵が短いと簡単に破られる、という体験です。

期待される出力: `鍵 0x5a -> FLAG{binary_hero}`

In [ ]:
def xor_cipher(data, key):
    return bytes(b ^ key for b in data)

secret = bytes.fromhex("1c161b1d213833343b282305323f283527")

for key in range(256):
    plain = xor_cipher(secret, key)
    if ____:                       # plain が b"FLAG" で始まるか
        print("鍵", hex(key), "->", plain.decode())
        break

## 対策2: 公開鍵暗号で盗聴を防ぐ

共通鍵の「鍵配送問題」を解決するのが 公開鍵暗号 です。鍵を 2つ 使います。

- 公開鍵: 誰に配ってもよい。暗号化 に使う
- 秘密鍵: 自分だけが持つ。復号 に使う

送りたい人は「受信者の公開鍵」で暗号化し、復号できるのは秘密鍵を持つ受信者だけ。鍵を配っても盗聴されません。

ここでは代表的な RSA を、小さな素数で手作りして仕組みを体験します。本物は数百桁の巨大な素数を使います。

- `n = p × q` は公開、`d = pow(e, -1, φ)` が秘密鍵
- 暗号化: `c = pow(m, e, n)` / 復号: `m = pow(c, d, n)`

In [ ]:
# --- 小さな素数で鍵を作る ---
p, q = 61, 53
n = p * q               # 3233。公開鍵の一部
phi = (p - 1) * (q - 1) # 3120。秘密で、他人に見せない
e = 17                  # 公開鍵の一部
d = pow(e, -1, phi)     # 秘密鍵 = mod phi での e の逆数。ここでは 2753
print("公開鍵 (e, n) =", (e, n), " / 秘密鍵 d =", d)

# --- 送信者: 受信者の公開鍵(e, n)で暗号化 ---
m = 65                  # 送りたい数。'A' の文字コード。m は n 未満である必要がある
c = pow(m, e, n)
print("暗号文:", c)      # 2790

# --- 受信者: 自分の秘密鍵 d で復号 ---
print("復号:", pow(c, d, n))   # 65 ← 秘密鍵を持つ本人だけが戻せる

## 練習問題6: RSA で暗号化・復号する

公開鍵 `(e, n)` と秘密鍵 `d` が与えられています。数 `m = 42` を 公開鍵で暗号化 し、その暗号文を 秘密鍵で復号 して、元の `42` に戻ることを確かめてください。

ヒント: 暗号化は `pow(m, e, n)`、復号は `pow(c, d, n)`。

期待される出力: `暗号文: 2557` / `復号: 42`

In [ ]:
e, n = 17, 3233   # 公開鍵
d = 2753          # 秘密鍵
m = 42

c = ____          # m を公開鍵で暗号化
print("暗号文:", c)
print("復号:", ____)   # c を秘密鍵で復号

## 対策3: ハッシュで改ざんを見抜く

SHA-256 などのハッシュ関数は、どんなデータからも決まった長さの「指紋」を作ります。ハッシュの3つの特性 を押さえましょう。

1. 一方向性、つまり不可逆性: データ→ハッシュは簡単だが、ハッシュ→元のデータは事実上不可能
2. 衝突困難性: 同じハッシュ値になる 別の入力を見つけるのが極めて難しい
3. 決定性・固定長: 同じ入力なら 必ず同じ出力。入力の長さによらず出力は固定長

この性質から、データの指紋を比べれば 改ざん を検知でき、パスワードも元を保存せず指紋だけで扱えます。

In [ ]:
import hashlib
# 決定性・固定長: 何を入れても64桁の16進、256ビット
print(hashlib.sha256(b"password").hexdigest())
print(hashlib.sha256(b"a very long message ...").hexdigest())

# わずかな違いでも指紋は全く別物。衝突困難性・一方向性を支える性質
print(hashlib.sha256(b"password").hexdigest()[:16], "...")
print(hashlib.sha256(b"passworE").hexdigest()[:16], "...  ← 1文字違うだけで激変")

## 練習問題7: ハッシュで改ざんを検知する

受信メッセージ `received` のSHA-256を計算し、送信側が付けた指紋 `expected` と一致するか判定してください。

ヒント: `hashlib.sha256(文字列.encode()).hexdigest()` を `==` で比較。

動作確認: そのままなら「改ざんなし」。`received` を書き換えると「改ざんあり！」。

In [ ]:
import hashlib
message = "振込先: 口座A"
message_bytes = message.encode()
expected = hashlib.sha256(message_bytes).hexdigest()   # 送信側の指紋

received = "振込先: 口座A"   # ← 書き換えると改ざんを再現できる
actual = ____                # received の SHA-256 の hexdigest
if ____:                     # actual と expected が一致するか
    print("改ざんなし")
else:
    print("改ざんあり！")

## 対策4: デジタル署名で改ざんとなりすましを防ぐ

公開鍵暗号を逆向きに使う と、本人証明、つまり署名になります。

- 署名: 送信者が 自分の秘密鍵 で作る。秘密鍵を持つ本人にしか作れない
- 検証: 受信者が 送信者の公開鍵 で確かめる

実際には、メッセージそのものではなく メッセージのハッシュ に署名します。こうすると、

- 中身が 改ざん されればハッシュが変わり、検証が失敗する → 改ざん検知
- 署名は秘密鍵がないと作れない → なりすまし防止

の両方が同時に実現できます。

In [ ]:
import hashlib
# 公開鍵暗号のセルで作った n, e, d をそのまま使う

message = b"give Bob 100 coins"
message_hash = hashlib.sha256(message).hexdigest()
digest = int(message_hash, 16) % n   # ハッシュを n 未満に

# 送信者: 秘密鍵 d で署名
signature = pow(digest, d, n)

# 受信者: 送信者の公開鍵 e で検証。自分でもハッシュを計算し直して比べる
check_hash = hashlib.sha256(message).hexdigest()
check = int(check_hash, 16) % n
print("正しい署名の検証:", pow(signature, e, n) == check)   # True = 本人・改ざんなし

# もしメッセージが改ざんされていたら…
tampered_hash = hashlib.sha256(b"give Bob 900 coins").hexdigest()
tampered = int(tampered_hash, 16) % n
print("改ざんされた場合:", pow(signature, e, n) == tampered) # False = 検知！

## 練習問題8: 総仕上げ、パケット復号ツールを作る

ゲームサーバから、XORでマスクされたパケット と、その 完全性ハッシュ が届きました。前半・後半で学んだことを全部使って中身を読み取ります。

届いたもの:
- `masked` … XORマスク済みパケット。16進文字列
- `key` … 共通鍵のマスク鍵 `0x3c`
- `expected` … 改ざん検知用の、元パケットのSHA-256

次の3ステップを行う関数 `decode_packet` を完成させてください。

1. マスクを解く: `masked` を鍵 `key` でXORして元のバイト列に戻す
2. 改ざん検知: 戻したバイト列のSHA-256が `expected` と一致するか。違えば「改ざんあり」で終了
3. 中身を読む: `struct.unpack(">BHhh", ...)` で id・x・y を取り出す

期待される出力:
```
改ざんなし
ID=7 x=100 y=-20
```

In [ ]:
import hashlib, struct

def xor_cipher(data, key):
    return bytes(b ^ key for b in data)

masked = "3d3c3b3c58c3d0"
key = 0x3c
expected = "734e0eb454532c0178e16046e5e3354d6a1a9230721e689a9296c8ee7e2a9fbe"

def decode_packet(masked, key, expected):
    raw = ____                       # 1) masked(16進)をバイト列にし、key でXORして解く
    if ____:                         # 2) raw の SHA-256 が expected と一致するか
        print("改ざんなし")
    else:
        print("改ざんあり！")
        return
    version, pid, x, y = ____        # 3) raw を ">BHhh" で unpack
    print("ID={} x={} y={}".format(pid, x, y))

decode_packet(masked, key, expected)

# おつかれさまでした

バイナリが読めると、情報セキュリティの世界が見えてきます。通信を脅かす 盗聴・改ざん・なりすまし に対して、

- 盗聴 → 暗号化: XOR などの共通鍵は速いが鍵配送が課題、RSA などの公開鍵は鍵を配れる
- 改ざん → ハッシュ: 一方向性・衝突困難性・決定性という3つの特性で指紋を作り、すり替えを見抜く
- なりすまし＋改ざん → デジタル署名: ハッシュに秘密鍵で署名し、公開鍵で検証する

という対策を、すべて手を動かして体験しました。

次回の ネットワークプログラミング回 では、これらのバイト列が実際にネットワークを流れる様子を HTTP と WebSocket で扱います。そして最後の 対戦ゲーム回 で、今日作った「パケットを読み解く力」を使って全員で競います。